In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [17]:
df = con.execute("""
                 SELECT *
                 FROM (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS ROW
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2026-02-09'
                ) WHERE ROW = 1
                 """).fetchdf()
df.head(6)

,NATBR,MAKTX_produto,WERKS,AINS,LABST,nome_arquivo,data_ingestao,ROW
0,10004,SERRA,BT50,100.0,200,z0019_2.csv,2026-02-09 22:00:11.558456,1
1,10001,PARAFUSO,BT10,100.0,100,z0019_1.csv,2026-02-09 21:54:46.894980,1
2,10002,MARTELO,BT50,100.0,1500,z0019_1.csv,2026-02-09 21:54:46.894980,1
3,10005,MACHADO,BT50,100.0,100,z0019_2.csv,2026-02-09 22:00:11.558456,1
4,10003,PREGO,BT10,100.0,60,z0019_2.csv,2026-02-09 22:00:11.558456,1


In [25]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'ROW'])
df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX_produto': 'nome_produto', 'WERKS': 'categoria', 'AINS': 'fornecedor', 'LABST': 'preco'})
df_final.head(6)

,id,nome_produto,categoria,fornecedor,preco
0,10004,SERRA,BT50,100.0,200
1,10001,PARAFUSO,BT10,100.0,100
2,10002,MARTELO,BT50,100.0,1500
3,10005,MACHADO,BT50,100.0,100
4,10003,PREGO,BT10,100.0,60


In [43]:
df2 = df_final
df2 = df2.astype({'id': int,
                  'nome_produto': str,
                  'categoria': str,
                  'fornecedor': int,
                  'preco': float}
                )

## df2.dtypes  ## mostra os tipos de dados de cada coluna do dataframe df2
df2.head(6)

,id,nome_produto,categoria,fornecedor,preco
0,10004,SERRA,BT50,100,200.0
1,10001,PARAFUSO,BT10,100,100.0
2,10002,MARTELO,BT50,100,1500.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [44]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT,
            nome_produto TEXT(255),
            categoria TEXT(255),
            fornecedor BIGINT,
            preco FLOAT
         )
""")
            

In [47]:
df2.head()

,id,nome_produto,categoria,fornecedor,preco
0,10004,SERRA,BT50,100,200.0
1,10001,PARAFUSO,BT10,100,100.0
2,10002,MARTELO,BT50,100,1500.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [49]:
con.execute("""INSERT INTO produtos SELECT * FROM df2""") ## insere os dados do dataframe df2 na tabela produtos

In [50]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head()


,id,nome_produto,categoria,fornecedor,preco
0,10004,SERRA,BT50,100,200.0
1,10001,PARAFUSO,BT10,100,100.0
2,10002,MARTELO,BT50,100,1500.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [51]:
con.close()